In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [13]:
df=pd.read_csv(r"C:\Users\user\OneDrive\Documents\D_I_G\ML\Timeseries\Forecasting_for_Action\interpolations_filled_NaN_train.csv")
df['date']=pd.to_datetime(df['date'],yearfirst=True)
df=df.sort_values('id')

In [14]:
df.sample()

,id,date,country,store,product,num_sold,1st_of_month,year,day,month,bi_monthly,sort_col,month_int,day_of_month,day_of_year,week,variable,6_clusters,4_clusters,is_stationary
175336,175368,2015-05-03,Kenya,Discount Stickers,Kerneler,9.0,2015-05-01,2015,3,May,May-1,1,5,3,123,2015-04-27/2015-05-03,Kenya\nDiscount Stickers\nKerneler,1,0,False


In [15]:
from prophet import Prophet
ts=Prophet()

In [16]:
df['preds']=np.nan

In [19]:
preds={}
for i in df['variable'].unique():
    ts=Prophet()
    data=df.loc[df['variable']==i]
    data=data[['date','num_sold']].rename(columns={'date':'ds','num_sold':'y'})
    ts.fit(data)
    forecast=ts.make_future_dataframe(periods=1095)
    preds[i]=ts.predict(forecast)[['ds','yhat']]

for key in preds.keys():
    print(key)

20:20:36 - cmdstanpy - INFO - Chain [1] start processing
20:20:37 - cmdstanpy - INFO - Chain [1] done processing
20:20:37 - cmdstanpy - INFO - Chain [1] start processing
20:20:38 - cmdstanpy - INFO - Chain [1] done processing
20:20:38 - cmdstanpy - INFO - Chain [1] start processing
20:20:39 - cmdstanpy - INFO - Chain [1] done processing
20:20:39 - cmdstanpy - INFO - Chain [1] start processing
20:20:40 - cmdstanpy - INFO - Chain [1] done processing
20:20:40 - cmdstanpy - INFO - Chain [1] start processing
20:20:40 - cmdstanpy - INFO - Chain [1] done processing
20:20:41 - cmdstanpy - INFO - Chain [1] start processing
20:20:42 - cmdstanpy - INFO - Chain [1] done processing
20:20:42 - cmdstanpy - INFO - Chain [1] start processing
20:20:43 - cmdstanpy - INFO - Chain [1] done processing
20:20:43 - cmdstanpy - INFO - Chain [1] start processing
20:20:44 - cmdstanpy - INFO - Chain [1] done processing
20:20:44 - cmdstanpy - INFO - Chain [1] start processing
20:20:45 - cmdstanpy - INFO - Chain [1]

Canada
Discount Stickers
Holographic Goose
Canada
Discount Stickers
Kaggle
Canada
Discount Stickers
Kaggle Tiers
Canada
Discount Stickers
Kerneler
Canada
Discount Stickers
Kerneler Dark Mode
Canada
Stickers for Less
Holographic Goose
Canada
Stickers for Less
Kaggle
Canada
Stickers for Less
Kaggle Tiers
Canada
Stickers for Less
Kerneler
Canada
Stickers for Less
Kerneler Dark Mode
Canada
Premium Sticker Mart
Holographic Goose
Canada
Premium Sticker Mart
Kaggle
Canada
Premium Sticker Mart
Kaggle Tiers
Canada
Premium Sticker Mart
Kerneler
Canada
Premium Sticker Mart
Kerneler Dark Mode
Finland
Discount Stickers
Holographic Goose
Finland
Discount Stickers
Kaggle
Finland
Discount Stickers
Kaggle Tiers
Finland
Discount Stickers
Kerneler
Finland
Discount Stickers
Kerneler Dark Mode
Finland
Stickers for Less
Holographic Goose
Finland
Stickers for Less
Kaggle
Finland
Stickers for Less
Kaggle Tiers
Finland
Stickers for Less
Kerneler
Finland
Stickers for Less
Kerneler Dark Mode
Finland
Premium Stic

In [42]:
display(preds["Singapore\nPremium Sticker Mart\nKerneler Dark Mode"].head())
display(preds["Singapore\nPremium Sticker Mart\nKerneler Dark Mode"].tail())

,ds,yhat
0,2010-01-01,900.234633
1,2010-01-02,962.742532
2,2010-01-03,1078.950011
3,2010-01-04,844.827604
4,2010-01-05,849.530913


,ds,yhat
3647,2019-12-27,1295.610530
3648,2019-12-28,1364.464066
3649,2019-12-29,1487.238766
3650,2019-12-30,1259.775727
3651,2019-12-31,1271.098983


In [52]:
sub_form=pd.read_csv(r"C:\Users\user\OneDrive\Documents\D_I_G\ML\Timeseries\Forecasting_for_Action\sample_submission.csv")
sub_form

,id,num_sold
0,230130,100
1,230131,100
2,230132,100
3,230133,100
4,230134,100
...,...,...
98545,328675,100
98546,328676,100
98547,328677,100
98548,328678,100


In [35]:
end_true={}
for i in df['variable'].unique():
    end_true[i]=int(df.loc[df['variable']==i].sort_values('id')['id'].tail(1).values)
len(end_true)

90

In [53]:
ids=np.array(range(90,(1095*90)+1,90))
ids[-10:]

array([97740, 97830, 97920, 98010, 98100, 98190, 98280, 98370, 98460,
       98550])

In [54]:
res=pd.DataFrame()
for k,v in preds.items():
    id_col=ids+end_true[k]
    v=v.iloc[:1095,:]
    v['id']=id_col
    res=pd.concat([res,v[['id','yhat']]],ignore_index=True)





In [55]:
res=res.sort_values('id').reset_index(drop=True)
res=res.rename(columns={'yhat':'num_sold'})
res.to_csv(r"C:\Users\user\OneDrive\Documents\D_I_G\ML\Timeseries\Forecasting_for_Action\submission_Prophet_FFA.csv",index=False)


In [56]:
res

,id,num_sold
0,230130,0.000000
1,230131,746.649882
2,230132,668.600035
3,230133,339.654544
4,230134,383.657869
...,...,...
98545,328675,384.653145
98546,328676,2469.406439
98547,328677,1969.470454
98548,328678,1108.924518
